# PHI De-Identification Accelerator — One-Click Launcher

> ⚠️ **SYNTHETIC DATA ONLY.** This launcher stands up the accelerator on **synthetic**
> Epic-shaped data (Caboodle + Clarity). It is a reference / blueprint pattern, **not** a certified
> de-identification service. Before any real PHI, work through
> [`docs/pre_real_phi_checklist.md`](docs/pre_real_phi_checklist.md).

**What this does (Run All, ~5–10 min):** automates every manual step in the
[QUICKSTART](QUICKSTART.md) so you don't have to click through the portal:

1. Uses **the workspace you imported this notebook into as the Raw workspace** (no throwaway
   workspace) and creates the other **two** (Analytics / Vault) on the same capacity.
2. Creates a **Lakehouse** in each (`lh_raw`, `lh_analytics`, `lh_vault`).
3. Downloads this repo from GitHub and uploads `src/` + `config/` to
   `Files/accelerator/` in each Lakehouse.
4. Uploads the 37 synthetic CSVs to the Raw Lakehouse — Caboodle (13) to
   `Files/raw/caboodle_provider/` and Clarity (24) to `Files/raw/clarity/`.
5. **Imports the notebooks** into the correct workspaces, auto-patching the
   cross-workspace config and binding each notebook's **default lakehouse** — no manual GUID edits, nothing to attach by hand.

**After it finishes** you set the tokenization pepper and run the notebooks in order
(the final cell prints the exact sequence). The manual QUICKSTART path still works if you
prefer to click through it yourself — this launcher is an **option**, not a replacement.

### How to run this
1. **Create a new workspace** and name it **`PHI-Raw`** — this becomes your Raw / PHI workspace.
   Make sure it's on an **F2+ / Trial** capacity.
2. **Import this notebook into that `PHI-Raw` workspace** (Workspace → New item → Import notebook).
3. Click **Run All**. The launcher keeps `PHI-Raw` as Raw and creates the other two workspaces
   (`PHI-Analytics`, `PHI-Vault`) for you — no extra/throwaway workspace.

### Prerequisites
- You must be able to **create workspaces** (Fabric admin setting: *Users can create workspaces*).
- Internet access to `github.com` (or fork to an internal location and set `GITHUB_OWNER`).

## 1. Configuration
Edit only if you want non-default names or a specific capacity. Defaults just work.

In [ ]:
# ---- GitHub source (public repo — no token needed) ----
GITHUB_OWNER = "rasgiza"
GITHUB_REPO = "fabric-phi-deidentification-accelerator"
GITHUB_BRANCH = "main"
GITHUB_TOKEN = ""  # only needed if you fork to a PRIVATE repo

# ---- Workspace + lakehouse names (created if missing, reused if present) ----
WS_RAW = "PHI-Raw"
WS_ANALYTICS = "PHI-Analytics"
WS_VAULT = "PHI-Vault"
LH_RAW = "lh_raw"
LH_ANALYTICS = "lh_analytics"
LH_VAULT = "lh_vault"

# ---- Capacity: leave blank to auto-pick the first ACTIVE capacity you can see ----
CAPACITY_ID = ""

# ---- Behaviour flags ----

# True  = use THIS notebook's workspace as Raw (nothing extra is created)
# False = also create WS_RAW
USE_CURRENT_WORKSPACE = True

# True = put all 3 lakehouses in ONE workspace.
# Quick for a sandbox, but it collapses the isolation boundary the demo is meant to show:
# Raw / Analytics / Vault exist as separate workspaces so identified data, safe data, and
# the re-identification key live under three different access-control surfaces.
SINGLE_WORKSPACE = False

# Upload the synthetic source CSVs to lh_raw (Caboodle: 13 tables, Clarity: 24 tables)
UPLOAD_DATA = True

# Import + auto-patch the pipeline notebooks into their workspaces
IMPORT_NOTEBOOKS = True

# Also import 03_gold_star — the "before" PHI-in-Gold notebook used in Demo Act 1
INCLUDE_BEFORE = True

print(
    f"Config loaded. Raw = {WS_RAW} | Analytics = {WS_ANALYTICS} | Vault = {WS_VAULT} "
    f"| single-workspace = {SINGLE_WORKSPACE}"
)

## 2. Helpers (Fabric REST + OneLake)

In [ ]:
import base64
import io
import json
import os
import re
import shutil
import tempfile
import time
import zipfile

import notebookutils
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

FABRIC_API = "https://api.fabric.microsoft.com/v1"

# (connect, read). Never call a remote API without a timeout: the default is *no* timeout,
# so one stalled connection hangs the notebook -- and the Spark session -- indefinitely.
HTTP_TIMEOUT = (10, 120)

_TOKEN = notebookutils.credentials.getToken("pbi")
H = {"Authorization": f"Bearer {_TOKEN}", "Content-Type": "application/json"}


def _build_session():
    """One retrying session for every Fabric API call.

    This launcher makes ~20 calls and is only PARTIALLY idempotent -- workspace and
    lakehouse creation are safe to repeat, but notebook import is not, so a mid-run
    failure leaves a half-built deployment that cannot simply be re-run. Fabric throttles
    with 429 + Retry-After and its gateway occasionally returns 502/503/504.

    Retries on STATUS only (``read=0``): those codes mean the request was rejected or never
    reached the service, so replaying it is safe. A read timeout AFTER the server accepted
    a POST is deliberately not retried -- that would risk creating the item twice.
    """
    retry = Retry(
        total=5,
        connect=3,
        read=0,
        status=5,
        backoff_factor=1.5,
        status_forcelist=(429, 502, 503, 504),
        respect_retry_after_header=True,
        raise_on_status=False,
    )
    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=retry))
    session.headers.update(H)
    return session


S = _build_session()


def _lro(resp, what="operation"):
    """Return the created/updated resource, following an async (202) Location if needed."""
    if resp.status_code in (200, 201):
        return resp.json() if resp.text else {}
    if resp.status_code == 202:
        op = resp.headers.get("Location")
        for _ in range(60):
            time.sleep(3)
            st = S.get(op, timeout=HTTP_TIMEOUT)
            state = st.json().get("status") if st.text else None
            if state == "Succeeded":
                res = S.get(op + "/result", timeout=HTTP_TIMEOUT)
                return res.json() if res.text else {}
            if state == "Failed":
                raise RuntimeError(f"{what} failed: {st.text}")
        raise TimeoutError(f"{what} did not complete in time")
    raise RuntimeError(f"{what} HTTP {resp.status_code}: {resp.text}")


def pick_capacity():
    if CAPACITY_ID:
        return CAPACITY_ID
    caps = S.get(f"{FABRIC_API}/capacities", timeout=HTTP_TIMEOUT).json().get("value", [])
    active = [c for c in caps if c.get("state", "").lower() == "active"]
    if not active:
        raise RuntimeError("No ACTIVE capacity found. Set CAPACITY_ID in the config cell.")
    print(f"Using capacity: {active[0]['displayName']} ({active[0]['id']})")
    return active[0]["id"]


def get_or_create_workspace(name, capacity_id):
    existing = S.get(f"{FABRIC_API}/workspaces", timeout=HTTP_TIMEOUT).json().get("value", [])
    for w in existing:
        if w["displayName"] == name:
            print(f"  • workspace exists: {name} ({w['id']})")
            S.post(
                f"{FABRIC_API}/workspaces/{w['id']}/assignToCapacity",
                json={"capacityId": capacity_id},
                timeout=HTTP_TIMEOUT,
            )
            return w["id"]
    body = {"displayName": name, "capacityId": capacity_id}
    w = _lro(
        S.post(f"{FABRIC_API}/workspaces", json=body, timeout=HTTP_TIMEOUT),
        f"create workspace {name}",
    )
    print(f"  • workspace created: {name} ({w['id']})")
    return w["id"]


def get_or_create_lakehouse(ws_id, name):
    existing = (
        S.get(f"{FABRIC_API}/workspaces/{ws_id}/lakehouses", timeout=HTTP_TIMEOUT)
        .json()
        .get("value", [])
    )
    for lh in existing:
        if lh["displayName"] == name:
            print(f"  • lakehouse exists: {name} ({lh['id']})")
            return lh["id"]
    lh = _lro(
        S.post(
            f"{FABRIC_API}/workspaces/{ws_id}/lakehouses",
            json={"displayName": name},
            timeout=HTTP_TIMEOUT,
        ),
        f"create lakehouse {name}",
    )
    print(f"  • lakehouse created: {name} ({lh['id']})")
    return lh["id"]


def onelake_files_root(ws_id, lh_id):
    # OneLake accepts EITHER GUIDs (no ".Lakehouse" suffix) OR friendly names
    # ("<name>.Lakehouse"). Mixing them -- "<guid>.Lakehouse" -- is rejected with
    # 400 FriendlyNameSupportDisabled. We always use GUIDs, so no suffix.
    return f"abfss://{ws_id}@onelake.dfs.fabric.microsoft.com/{lh_id}/Files"


def upload_dir(local_dir, abfss_dir):
    """Copy a local directory tree into OneLake Files/ and return the file count.

    OneLake returns HTTP 400 on a write whose PARENT directory does not exist yet,
    so every target directory is created with mkdirs() before its files are copied.
    """
    n = 0
    for root, _dirs, files in os.walk(local_dir):
        if "__pycache__" in root:
            continue
        rel_dir = os.path.relpath(root, local_dir).replace(os.sep, "/")
        target_dir = abfss_dir if rel_dir == "." else f"{abfss_dir}/{rel_dir}"
        notebookutils.fs.mkdirs(target_dir)
        for fn in files:
            if fn.endswith(".pyc"):
                continue
            lp = os.path.join(root, fn)
            notebookutils.fs.cp(f"file:{lp}", f"{target_dir}/{fn}", True)
            n += 1
    print(f"    uploaded {n} files -> {abfss_dir}")
    return n


print("Helpers ready.")

## 3. Download the repo from GitHub

In [ ]:
zip_url = f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}/archive/refs/heads/{GITHUB_BRANCH}.zip"
dl_headers = {"Authorization": f"token {GITHUB_TOKEN}"} if GITHUB_TOKEN else {}

print("Downloading", zip_url)
# Plain requests, NOT the Fabric session S -- this call must not carry the Fabric bearer token.
resp = requests.get(zip_url, headers=dl_headers, timeout=HTTP_TIMEOUT)
resp.raise_for_status()
zf = zipfile.ZipFile(io.BytesIO(resp.content))

# mkdtemp() instead of a fixed "/tmp/..." path: a predictable name in a world-writable
# directory is a symlink / pre-creation attack surface, and two concurrent runs would
# otherwise clobber each other.
extract_root = tempfile.mkdtemp(prefix="phi_deid_repo_")

# Zip-slip guard. GitHub archives are trustworthy, but "we trust the source" is precisely
# the assumption that makes path-traversal extraction a recurring CVE -- and this notebook
# runs with permission to write across three workspaces. Validate members, then extract.
_root = os.path.realpath(extract_root)
for member in zf.namelist():
    if not os.path.realpath(os.path.join(_root, member)).startswith(_root + os.sep):
        shutil.rmtree(extract_root, ignore_errors=True)
        raise RuntimeError(f"Refusing to extract archive: unsafe member path {member!r}")
zf.extractall(extract_root)

# Discover the extracted folder rather than rebuilding its name from GITHUB_BRANCH.
# GitHub flattens slashes in archive folder names, so branch "feat/my-change" extracts to
# "<repo>-feat-my-change" and any guessed path is wrong for every branch with a "/" in it.
# GitHub archives always contain exactly one top-level directory.
_tops = [d for d in os.listdir(extract_root) if os.path.isdir(os.path.join(extract_root, d))]
if len(_tops) != 1:
    raise RuntimeError(f"Expected one top-level folder in the archive, found: {_tops}")
REPO = os.path.join(extract_root, _tops[0])
if not os.path.isdir(os.path.join(REPO, "src")):
    raise RuntimeError(
        f"src/ not found under {REPO} — check GITHUB_OWNER / GITHUB_REPO / GITHUB_BRANCH."
    )
print("Extracted to", REPO)


## 4. Create workspaces + lakehouses

In [ ]:
cap = pick_capacity()

print("Workspaces:")
if USE_CURRENT_WORKSPACE:
    _ctx = notebookutils.runtime.context
    ws_raw = _ctx.get("currentWorkspaceId") or _ctx.get("workspaceId")
    WS_RAW = (
        _ctx.get("currentWorkspaceName") or WS_RAW
    )  # use the real name for cross-workspace config patching
    print(f"  • using CURRENT workspace as Raw: {WS_RAW} ({ws_raw})")
else:
    ws_raw = get_or_create_workspace(WS_RAW, cap)
if SINGLE_WORKSPACE:
    ws_analytics = ws_vault = ws_raw
    WS_ANALYTICS = WS_VAULT = WS_RAW  # so config-patching points everything at the one workspace
    print("  (single-workspace mode: Analytics + Vault reuse the Raw workspace)")
else:
    ws_analytics = get_or_create_workspace(WS_ANALYTICS, cap)
    ws_vault = get_or_create_workspace(WS_VAULT, cap)

print("Lakehouses:")
lh_raw_id = get_or_create_lakehouse(ws_raw, LH_RAW)
lh_analytics_id = get_or_create_lakehouse(ws_analytics, LH_ANALYTICS)
lh_vault_id = get_or_create_lakehouse(ws_vault, LH_VAULT)

## 5. Upload code, config, and sample data

In [ ]:
# Upload src/ + config/ to Files/accelerator in EACH lakehouse (notebooks import from the attached default).
targets = {
    LH_RAW: (ws_raw, lh_raw_id),
    LH_ANALYTICS: (ws_analytics, lh_analytics_id),
    LH_VAULT: (ws_vault, lh_vault_id),
}
for name, (ws_id, lh_id) in targets.items():
    files_root = onelake_files_root(ws_id, lh_id)
    print(f"Uploading accelerator package to {name} ...")
    upload_dir(os.path.join(REPO, "src"), f"{files_root}/accelerator/src")
    upload_dir(os.path.join(REPO, "config"), f"{files_root}/accelerator/config")

# ── Synthetic source data → RAW lakehouse only ────────────────────────────────
# This registry MIRRORS the SOURCES registry in 01_bronze_ingest. The target folder names
# are a CONTRACT between the two: 01 reads Files/raw/<key>, so renaming a destination here
# gives you a silently empty ingest rather than an error.
#   caboodle → dimensional warehouse (13 tables)
#   clarity  → normalized transactional schema (24 tables)
# Two source schemas, one de-identification engine — adding the second was a config change,
# not a code change. That portability is the reason both are shipped.
SAMPLE_SOURCES = {
    "caboodle": {"local": "caboodle_provider", "target": "raw/caboodle_provider"},
    "clarity": {"local": "Clarity", "target": "raw/clarity"},
}

if UPLOAD_DATA:
    raw_files_root = onelake_files_root(ws_raw, lh_raw_id)
    for key, src_cfg in SAMPLE_SOURCES.items():
        local = os.path.join(REPO, "sample_data", src_cfg["local"])
        # The repo zip is the source of truth: a missing folder is a broken release, not a
        # user setting, so fail loudly here rather than half-loading the demo.
        if not os.path.isdir(local):
            raise RuntimeError(f"sample_data/{src_cfg['local']} not found in {REPO}")
        print(f"Uploading synthetic {key} CSVs to {LH_RAW} ...")
        upload_dir(local, f"{raw_files_root}/{src_cfg['target']}")

## 6. Import notebooks (auto-patched — no manual GUID edits)

In [ ]:
def _patch_assignment(src, var, value):
    """Rewrite a top-level `VAR = \"...\"` assignment to VAR = \"value\"."""
    pat = re.compile(rf'^(\s*{var}\s*=\s*)"[^"]*"', re.M)
    return pat.sub(lambda m: f'{m.group(1)}"{value}"', src)


def _patch_notebook_json(nb_json, patches):
    for cell in nb_json.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        src = "".join(cell.get("source", []))
        new = src
        for var, val in patches.items():
            new = _patch_assignment(new, var, val)
        if new != src:
            cell["source"] = new.splitlines(keepends=True)
    return nb_json


def _attach_default_lakehouse(nb_json, lh_id, lh_name, ws_id):
    """Bind the default lakehouse into the notebook definition itself.

    Every notebook here resolves data two ways: `spark.table("bronze_x")` and the
    `/lakehouse/default/Files/accelerator` import path. Both need a DEFAULT lakehouse, and
    a freshly imported notebook has none -- so without this the adopter must remember to
    attach the right lakehouse by hand, to the right notebook, before each run. Getting it
    wrong is quiet: the notebook attaches to *a* lakehouse and reads an empty catalog.

    Writing the binding into the definition also makes the pipeline runnable headlessly
    (REST job / scheduled Data Factory pipeline), not just by a human clicking Run All.
    """
    meta = nb_json.setdefault("metadata", {})
    meta.setdefault("dependencies", {})["lakehouse"] = {
        "default_lakehouse": lh_id,
        "default_lakehouse_name": lh_name,
        "default_lakehouse_workspace_id": ws_id,
    }
    return nb_json


def _find_notebook(ws_id, display_name):
    r = S.get(f"{FABRIC_API}/workspaces/{ws_id}/items?type=Notebook", timeout=HTTP_TIMEOUT)
    r.raise_for_status()
    for item in r.json().get("value", []):
        if item["displayName"] == display_name:
            return item["id"]
    return None


def import_notebook(ws_id, display_name, local_ipynb, patches=None, lakehouse=None):
    with open(local_ipynb, encoding="utf-8") as fh:
        nb = json.load(fh)
    if patches:
        nb = _patch_notebook_json(nb, patches)
    if lakehouse:
        nb = _attach_default_lakehouse(nb, *lakehouse)
    payload = base64.b64encode(json.dumps(nb).encode("utf-8")).decode("ascii")
    body = {
        "displayName": display_name,
        "definition": {
            "format": "ipynb",
            "parts": [
                {
                    "path": "notebook-content.ipynb",
                    "payload": payload,
                    "payloadType": "InlineBase64",
                }
            ],
        },
    }
    # Update in place when the notebook already exists. Creating blindly returns HTTP 409
    # ItemDisplayNameAlreadyInUse, which made this launcher effectively single-use: any
    # partial run -- wrong capacity, expired token, or a workspace that already held an
    # older copy of the accelerator -- leaves names behind, and then the obvious recovery
    # ("fix the setting, run it again") fails on the very first notebook.
    existing = _find_notebook(ws_id, display_name)
    if existing:
        _lro(
            S.post(
                f"{FABRIC_API}/workspaces/{ws_id}/items/{existing}/updateDefinition",
                json={"definition": body["definition"]},
                timeout=HTTP_TIMEOUT,
            ),
            f"update notebook {display_name}",
        )
        verb = "updated "
    else:
        _lro(
            S.post(f"{FABRIC_API}/workspaces/{ws_id}/notebooks", json=body, timeout=HTTP_TIMEOUT),
            f"import notebook {display_name}",
        )
        verb = "imported"
    print(f"  • {verb} {display_name}  (default lakehouse: {lakehouse[1] if lakehouse else 'none'})")


if IMPORT_NOTEBOOKS:
    nbdir = os.path.join(REPO, "notebooks")
    # Cross-workspace config injected so the notebooks resolve each other with NO manual edits.
    # GUIDs (not friendly names) -- some tenants have OneLake friendly-name resolution disabled.
    patch_03b = {"SOURCE_WORKSPACE": ws_raw, "SOURCE_LAKEHOUSE": lh_raw_id}
    patch_reid = {
        "RAW_WORKSPACE": ws_raw,
        "RAW_LAKEHOUSE": lh_raw_id,
        "ANALYTICS_WORKSPACE": ws_analytics,
        "ANALYTICS_LAKEHOUSE": lh_analytics_id,
    }

    # Each notebook is bound to the lakehouse it is SUPPOSED to read/write, which is also
    # the isolation boundary: Raw notebooks cannot accidentally default to the safe
    # lakehouse, and Analytics notebooks cannot accidentally default to the PHI one.
    lh_for_raw = (lh_raw_id, LH_RAW, ws_raw)
    lh_for_analytics = (lh_analytics_id, LH_ANALYTICS, ws_analytics)
    lh_for_vault = (lh_vault_id, LH_VAULT, ws_vault)

    print("Raw workspace:")
    import_notebook(ws_raw, "01_bronze_ingest",
                    os.path.join(nbdir, "01_bronze_ingest.ipynb"), lakehouse=lh_for_raw)
    import_notebook(ws_raw, "02_silver_conform",
                    os.path.join(nbdir, "02_silver_conform.ipynb"), lakehouse=lh_for_raw)
    import_notebook(ws_raw, "02b_silver_deid",
                    os.path.join(nbdir, "02b_silver_deid.ipynb"), lakehouse=lh_for_raw)
    if INCLUDE_BEFORE:
        import_notebook(ws_raw, "03_gold_star",
                        os.path.join(nbdir, "03_gold_star.ipynb"), lakehouse=lh_for_raw)

    print("Analytics workspace:")
    import_notebook(
        ws_analytics,
        "03b_gold_safe_analytics",
        os.path.join(nbdir, "03b_gold_safe_analytics.ipynb"),
        patch_03b,
        lakehouse=lh_for_analytics,
    )
    import_notebook(ws_analytics, "NB_scorecard",
                    os.path.join(nbdir, "NB_scorecard.ipynb"), lakehouse=lh_for_analytics)

    print("Vault workspace:")
    import_notebook(
        ws_vault, "NB_reidentify", os.path.join(nbdir, "NB_reidentify.ipynb"),
        patch_reid, lakehouse=lh_for_vault,
    )


## 7. Done — next steps

In [ ]:
print("=" * 70)
print("DEPLOYMENT COMPLETE")
print("=" * 70)
print(f"Raw workspace       : {WS_RAW}       (lakehouse {LH_RAW})")
print(f"Analytics workspace : {WS_ANALYTICS} (lakehouse {LH_ANALYTICS})")
print(f"Vault workspace     : {WS_VAULT}     (lakehouse {LH_VAULT})")
print()
print("NEXT STEPS")
print("-" * 70)
print("1. Attach the RIGHT default lakehouse to each notebook before running it")
print("   (Lakehouse explorer -> Add -> Existing lakehouse):")
print(f"     Raw notebooks       -> {LH_RAW}")
print(f"     Analytics notebooks -> {LH_ANALYTICS}")
print(f"     Vault notebook      -> {LH_VAULT}")
print("2. Run the notebooks in order:")
print("     Raw:       01_bronze_ingest -> 02_silver_conform -> 02b_silver_deid")
print("     Analytics: 03b_gold_safe_analytics -> NB_scorecard   (expect PASS: 0/18)")
print("     Vault:     NB_reidentify  (only for a governed, approved re-identification)")
print()
print("Pepper: nothing to do for the SYNTHETIC demo -- 02b_silver_deid and NB_reidentify")
print("each set the same fixed high-entropy DEMO_PEPPER themselves. For real PHI, delete")
print(
    "that cell and use Key Vault (set PHI_DEID_KEYVAULT_URL); see docs/pre_real_phi_checklist.md."
)
print()
print("Optional 'before' demo (Act 1): run 03_gold_star in Raw to show PHI reaching Gold.")
print("Full guided demo: docs/demo_runbook.md")